# Module 4: Model Selection - FE vs RE

## Learning Objectives

By completing this notebook, you will:
1. Understand when to use Fixed Effects vs Random Effects models
2. Implement and interpret the Hausman test for model selection
3. Apply specification tests for panel data models
4. Compare pooled OLS, FE, and RE estimates systematically
5. Make informed model selection decisions based on data structure and assumptions

## Prerequisites

- Module 2 (Fixed Effects) completed
- Module 3 (Random Effects) completed
- Understanding of hypothesis testing

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS, RandomEffects
from linearmodels.panel.results import compare

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

In [ ]:
section_divider("1. FE vs RE: Conceptual Framework")

In [ ]:
learning_objectives(["Understand when to use Fixed Effects vs Random Effects models", "Implement and interpret the Hausman test for model selection", "Apply specification tests for panel data models", "Compare pooled OLS, FE, and RE estimates systematically", "Make informed model selection decisions based on data structure and assumptions"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. FE vs RE: Conceptual Framework

### Fixed Effects Model

$$y_{it} = \alpha_i + X_{it}\beta + \epsilon_{it}$$

**Assumption:** $Cov(\alpha_i, X_{it})$ can be $\neq 0$ (correlation allowed)

**Advantage:** Controls for all time-invariant confounders

**Disadvantage:** Cannot estimate effects of time-invariant variables

### Random Effects Model

$$y_{it} = \beta_0 + X_{it}\beta + (\alpha_i + \epsilon_{it})$$

**Assumption:** $Cov(\alpha_i, X_{it}) = 0$ (strict exogeneity)

**Advantage:** More efficient, can estimate time-invariant effects

**Disadvantage:** Inconsistent if assumption violated

### Key Question

> **Is $Cov(\alpha_i, X_{it}) = 0$ plausible?**
>
> - If YES → Use Random Effects (more efficient)
> - If NO → Use Fixed Effects (consistent)
> - If UNSURE → Test with Hausman test

In [ ]:
section_divider("2. Generate Data with Controlled Correlation")

## 2. Generate Data with Controlled Correlation

We'll create two datasets:
1. **No correlation:** RE assumption holds
2. **Correlation present:** FE needed

In [ ]:
def generate_panel_data(n_entities=100, n_periods=10, alpha_x_corr=0.0, beta_true=2.0):
    """
    Generate panel data with controlled correlation between fixed effects and X.
    
    Parameters
    ----------
    n_entities : int
    n_periods : int
    alpha_x_corr : float
        Correlation between alpha_i and X (0 = RE holds, non-zero = FE needed)
    beta_true : float
        True causal effect
    
    Returns
    -------
    pd.DataFrame
    """
    # Entity fixed effects
    alpha_i = np.random.randn(n_entities) * 10 + 50
    
    # Generate X with specified correlation to alpha
    data = []
    for i in range(n_entities):
        # Base X depends on alpha (creates correlation)
        base_x = 10 + alpha_x_corr * (alpha_i[i] - 50) / 10
        
        for t in range(n_periods):
            # X varies over time
            x_it = base_x + np.random.randn() * 2
            
            # Y = alpha + beta * X + error
            y_it = alpha_i[i] + beta_true * x_it + np.random.randn() * 5
            
            data.append({
                'entity': i + 1,
                'time': t + 1,
                'y': y_it,
                'x': x_it,
                'alpha_true': alpha_i[i]
            })
    
    return pd.DataFrame(data)

# Dataset 1: No correlation (RE assumption holds)
df_re_valid = generate_panel_data(alpha_x_corr=0.0)
print("Dataset 1: RE assumption VALID (no correlation)")
print(f"  N = {df_re_valid['entity'].nunique()}, T = {df_re_valid['time'].nunique()}")
print(f"  True beta: 2.0\n")

# Dataset 2: Correlation present (FE needed)
df_fe_needed = generate_panel_data(alpha_x_corr=0.8)
print("Dataset 2: RE assumption VIOLATED (correlation = 0.8)")
print(f"  N = {df_fe_needed['entity'].nunique()}, T = {df_fe_needed['time'].nunique()}")
print(f"  True beta: 2.0")

### Verify Correlation Structure

In [ ]:
# Compute entity means to check correlation with alpha
def check_correlation_structure(df):
    entity_data = df.groupby('entity').agg({
        'x': 'mean',
        'alpha_true': 'first'
    }).reset_index()
    
    corr = entity_data[['alpha_true', 'x']].corr().iloc[0, 1]
    return corr

corr1 = check_correlation_structure(df_re_valid)
corr2 = check_correlation_structure(df_fe_needed)

print(f"Correlation between alpha_i and X_bar_i:")
print(f"  Dataset 1 (RE valid): {corr1:.4f}")
print(f"  Dataset 2 (FE needed): {corr2:.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (df, title, corr) in enumerate([
    (df_re_valid, 'Dataset 1: No Correlation', corr1),
    (df_fe_needed, 'Dataset 2: Strong Correlation', corr2)
]):
    entity_data = df.groupby('entity').agg({'x': 'mean', 'alpha_true': 'first'})
    
    axes[idx].scatter(entity_data['x'], entity_data['alpha_true'], alpha=0.5)
    
    # Fit line
    z = np.polyfit(entity_data['x'], entity_data['alpha_true'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(entity_data['x'].min(), entity_data['x'].max(), 100)
    axes[idx].plot(x_line, p(x_line), 'r--', linewidth=2)
    
    axes[idx].set_xlabel('X (entity mean)', fontsize=11)
    axes[idx].set_ylabel('True Fixed Effect (alpha_i)', fontsize=11)
    axes[idx].set_title(f'{title}\nCorrelation: {corr:.3f}', fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
section_divider("3. Compare Pooled OLS, FE, and RE")

## 3. Compare Pooled OLS, FE, and RE

Let's estimate all three models on both datasets.

In [ ]:
def estimate_all_models(df, entity_col='entity', time_col='time'):
    """
    Estimate pooled OLS, FE, and RE on panel data.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    
    Returns
    -------
    dict
        Estimation results
    """
    # Prepare data
    df_panel = df.set_index([entity_col, time_col])
    
    # Pooled OLS
    model_pooled = PanelOLS(
        dependent=df_panel['y'],
        exog=df_panel[['x']],
        entity_effects=False
    )
    results_pooled = model_pooled.fit()
    
    # Fixed Effects
    model_fe = PanelOLS(
        dependent=df_panel['y'],
        exog=df_panel[['x']],
        entity_effects=True
    )
    results_fe = model_fe.fit()
    
    # Random Effects
    model_re = RandomEffects(
        dependent=df_panel['y'],
        exog=df_panel[['x']]
    )
    results_re = model_re.fit()
    
    return {
        'pooled': results_pooled,
        'fe': results_fe,
        're': results_re
    }

# Estimate on both datasets
print("Estimating models on both datasets...\n")
results_dataset1 = estimate_all_models(df_re_valid)
results_dataset2 = estimate_all_models(df_fe_needed)

### Compare Coefficient Estimates

In [ ]:
def compare_estimates(results_dict, true_beta=2.0, dataset_name=""):
    """
    Create comparison table of estimates.
    """
    comparison = []
    
    for model_name in ['pooled', 'fe', 're']:
        res = results_dict[model_name]
        beta_hat = res.params['x']
        se = res.std_errors['x']
        bias = beta_hat - true_beta
        
        comparison.append({
            'Model': model_name.upper(),
            'Estimate': beta_hat,
            'Std Error': se,
            'Bias': bias,
            'Bias %': (bias / true_beta) * 100
        })
    
    df_comp = pd.DataFrame(comparison)
    
    print(f"\n{'='*70}")
    print(f"{dataset_name}")
    print(f"True beta: {true_beta}")
    print(f"{'='*70}")
    print(df_comp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    
    return df_comp

comp1 = compare_estimates(results_dataset1, dataset_name="Dataset 1: No Correlation (RE Valid)")
comp2 = compare_estimates(results_dataset2, dataset_name="Dataset 2: Correlation Present (FE Needed)")

### Interpretation

**Dataset 1 (No Correlation):**
- All models approximately unbiased
- RE more efficient (smaller SE)
- FE also consistent but less efficient

**Dataset 2 (Correlation Present):**
- Pooled OLS and RE biased
- FE unbiased (controls for correlation)
- This is why we need the Hausman test!

In [ ]:
section_divider("4. The Hausman Test")

## 4. The Hausman Test

### Test Logic

**Null hypothesis:** $H_0: Cov(\alpha_i, X_{it}) = 0$ (RE is consistent and efficient)

**Alternative:** $H_a: Cov(\alpha_i, X_{it}) \neq 0$ (Only FE is consistent)

**Test statistic:**

$$H = (\hat{\beta}_{FE} - \hat{\beta}_{RE})' [Var(\hat{\beta}_{FE}) - Var(\hat{\beta}_{RE})]^{-1} (\hat{\beta}_{FE} - \hat{\beta}_{RE})$$

Under $H_0$: $H \sim \chi^2_k$ where $k$ = number of time-varying variables

**Decision:**
- Reject $H_0$ → Use FE
- Fail to reject → Use RE

In [ ]:
def hausman_test(results_fe, results_re):
    """
    Implement Hausman test for FE vs RE.
    
    Parameters
    ----------
    results_fe : linearmodels results
    results_re : linearmodels results
    
    Returns
    -------
    dict
        Test statistic, p-value, decision
    """
    # Get coefficients (only time-varying variables)
    beta_fe = results_fe.params
    beta_re = results_re.params
    
    # Get covariance matrices
    cov_fe = results_fe.cov
    cov_re = results_re.cov
    
    # Ensure same variables
    common_vars = beta_fe.index.intersection(beta_re.index)
    beta_fe = beta_fe[common_vars]
    beta_re = beta_re[common_vars]
    cov_fe = cov_fe.loc[common_vars, common_vars]
    cov_re = cov_re.loc[common_vars, common_vars]
    
    # Difference
    diff = beta_fe - beta_re
    
    # Covariance of difference
    cov_diff = cov_fe - cov_re
    
    # Test statistic
    try:
        h_stat = float(diff.T @ np.linalg.inv(cov_diff) @ diff)
        df = len(diff)
        p_value = 1 - stats.chi2.cdf(h_stat, df)
        
        decision = "Reject H0: Use FE" if p_value < 0.05 else "Fail to reject H0: Use RE"
        
        return {
            'test_statistic': h_stat,
            'df': df,
            'p_value': p_value,
            'decision': decision,
            'valid': True
        }
    except np.linalg.LinAlgError:
        return {
            'test_statistic': np.nan,
            'df': len(diff),
            'p_value': np.nan,
            'decision': 'Test failed (covariance matrix not positive definite)',
            'valid': False
        }

# Run Hausman test on both datasets
print("\nHausman Test Results")
print("="*70)

hausman1 = hausman_test(results_dataset1['fe'], results_dataset1['re'])
print(f"\nDataset 1 (No Correlation - RE should be preferred):")
print(f"  H-statistic: {hausman1['test_statistic']:.4f}")
print(f"  P-value: {hausman1['p_value']:.4f}")
print(f"  Decision: {hausman1['decision']}")

hausman2 = hausman_test(results_dataset2['fe'], results_dataset2['re'])
print(f"\nDataset 2 (Correlation Present - FE should be preferred):")
print(f"  H-statistic: {hausman2['test_statistic']:.4f}")
print(f"  P-value: {hausman2['p_value']:.4f}")
print(f"  Decision: {hausman2['decision']}")

### Interpretation

**Dataset 1:** Large p-value → Fail to reject → FE and RE similar → Use RE (more efficient)

**Dataset 2:** Small p-value → Reject → FE and RE differ → Use FE (only consistent estimator)

The Hausman test correctly identifies when correlation is present!

In [ ]:
section_divider("5. Practical Example: Firm R&D and Patents")

## 5. Practical Example: Firm R&D and Patents

Apply model selection to realistic economic data.

In [ ]:
# Generate firm-level panel data
n_firms = 150
n_years = 12

# Firm quality (unobserved, affects both R&D and patents)
firm_quality = np.random.randn(n_firms) * 20 + 100

data = []
for i in range(n_firms):
    # R&D spending correlated with firm quality
    base_rd = 50 + 0.3 * firm_quality[i] + np.random.randn() * 10
    
    for t in range(n_years):
        # R&D varies over time
        rd_it = base_rd + np.random.randn() * 8
        rd_it = max(10, rd_it)  # Non-negative
        
        # Patents = firm quality + true R&D effect + error
        patents_it = firm_quality[i] + 1.5 * rd_it + np.random.randn() * 15
        
        # Market size (exogenous time-varying)
        market_size = 100 + t * 2 + np.random.randn() * 5
        
        data.append({
            'firm_id': i + 1,
            'year': 2010 + t,
            'patents': patents_it,
            'rd_spending': rd_it,
            'market_size': market_size,
            'firm_quality': firm_quality[i]
        })

df_firms = pd.DataFrame(data)

print("Firm R&D and Patents Dataset")
print(f"  N = {n_firms} firms, T = {n_years} years")
print(f"  Total observations: {len(df_firms)}")
print(f"\nTrue causal effect of R&D on patents: 1.5")
print(f"\nFirst 10 rows:")
print(df_firms.head(10))

### Estimate All Models

In [ ]:
# Prepare panel data
df_firms_panel = df_firms.set_index(['firm_id', 'year'])

# Pooled OLS
model_pooled_firms = PanelOLS(
    dependent=df_firms_panel['patents'],
    exog=df_firms_panel[['rd_spending', 'market_size']],
    entity_effects=False
)
results_pooled_firms = model_pooled_firms.fit(cov_type='clustered', cluster_entity=True)

# Fixed Effects
model_fe_firms = PanelOLS(
    dependent=df_firms_panel['patents'],
    exog=df_firms_panel[['rd_spending', 'market_size']],
    entity_effects=True
)
results_fe_firms = model_fe_firms.fit(cov_type='clustered', cluster_entity=True)

# Random Effects
model_re_firms = RandomEffects(
    dependent=df_firms_panel['patents'],
    exog=df_firms_panel[['rd_spending', 'market_size']]
)
results_re_firms = model_re_firms.fit()

print("\nModel Comparison: Pooled OLS vs FE vs RE")
print("="*90)
print(f"{'Model':<15} {'RD Coef':>12} {'RD SE':>10} {'Market Coef':>12} {'Market SE':>10} {'R²':>8}")
print("="*90)

for name, res in [('Pooled OLS', results_pooled_firms), 
                   ('Fixed Effects', results_fe_firms), 
                   ('Random Effects', results_re_firms)]:
    rd_coef = res.params['rd_spending']
    rd_se = res.std_errors['rd_spending']
    market_coef = res.params['market_size']
    market_se = res.std_errors['market_size']
    r2 = res.rsquared if hasattr(res, 'rsquared') else res.rsquared_overall
    
    print(f"{name:<15} {rd_coef:>12.4f} {rd_se:>10.4f} {market_coef:>12.4f} {market_se:>10.4f} {r2:>8.4f}")

print("="*90)
print(f"True R&D effect: 1.50")
print("\nNote: Pooled OLS overestimates R&D effect (omitted variable bias from firm quality)")

### Apply Hausman Test

In [ ]:
# Hausman test
hausman_firms = hausman_test(results_fe_firms, results_re_firms)

print("\nHausman Test: FE vs RE")
print("="*70)
print(f"H-statistic: {hausman_firms['test_statistic']:.4f}")
print(f"Degrees of freedom: {hausman_firms['df']}")
print(f"P-value: {hausman_firms['p_value']:.4f}")
print(f"\nDecision (α = 0.05): {hausman_firms['decision']}")

if hausman_firms['p_value'] < 0.05:
    print("\nInterpretation:")
    print("  - Significant difference between FE and RE")
    print("  - Evidence of correlation between firm effects and R&D")
    print("  - Fixed Effects model is preferred")
    print("  - RE would produce biased estimates")
else:
    print("\nInterpretation:")
    print("  - No significant difference between FE and RE")
    print("  - No evidence of correlation")
    print("  - Random Effects model preferred (more efficient)")

## 6. Other Specification Tests

Beyond Hausman, other tests help with model selection.

### F-Test for Entity Fixed Effects

In [ ]:
def f_test_entity_effects(results_pooled, results_fe):
    """
    F-test for significance of entity fixed effects.
    
    H0: All entity effects are zero (pooled OLS is adequate)
    Ha: Entity effects are significant (FE needed)
    """
    # Residual sum of squares
    rss_pooled = results_pooled.resids.T @ results_pooled.resids
    rss_fe = results_fe.resids.T @ results_fe.resids
    
    # Degrees of freedom
    n_entities = len(results_fe.entity_info.index)  # Number of entities
    n_obs = results_fe.nobs
    k = len(results_fe.params)  # Number of X variables
    
    # F-statistic
    df1 = n_entities - 1
    df2 = n_obs - n_entities - k
    
    f_stat = ((rss_pooled - rss_fe) / df1) / (rss_fe / df2)
    p_value = 1 - stats.f.cdf(f_stat, df1, df2)
    
    return {
        'f_statistic': f_stat,
        'df1': df1,
        'df2': df2,
        'p_value': p_value
    }

f_test_result = f_test_entity_effects(results_pooled_firms, results_fe_firms)

print("\nF-Test for Entity Fixed Effects")
print("="*70)
print(f"H0: All entity fixed effects are zero (pooled OLS adequate)")
print(f"Ha: Entity fixed effects are significant\n")
print(f"F-statistic: {f_test_result['f_statistic']:.4f}")
print(f"Degrees of freedom: ({f_test_result['df1']}, {f_test_result['df2']})")
print(f"P-value: {f_test_result['p_value']:.4e}")

if f_test_result['p_value'] < 0.05:
    print("\n✅ Reject H0: Entity fixed effects are significant")
    print("   → Use panel model (FE or RE), not pooled OLS")
else:
    print("\n❌ Fail to reject H0: Entity effects not significant")
    print("   → Pooled OLS may be adequate")

### Breusch-Pagan LM Test for Random Effects

In [ ]:
def breusch_pagan_test(results_pooled, n_entities, n_periods):
    """
    Breusch-Pagan LM test for random effects.
    
    H0: Var(alpha_i) = 0 (no random effects, pooled OLS adequate)
    Ha: Var(alpha_i) > 0 (random effects present)
    """
    # Compute entity-specific residual sums
    resids = pd.Series(results_pooled.resids.values.flatten(), 
                      index=results_pooled.resids.index)
    entity_resid_means = resids.groupby(level=0).mean()
    
    # LM statistic
    numerator = n_periods * (entity_resid_means ** 2).sum()
    denominator = (resids ** 2).sum()
    
    lm_stat = (numerator / denominator) - 1
    lm_stat = (lm_stat ** 2) * (n_entities * n_periods) / (2 * (n_periods - 1))
    
    # P-value (chi-squared with 1 df)
    p_value = 1 - stats.chi2.cdf(lm_stat, 1)
    
    return {
        'lm_statistic': lm_stat,
        'p_value': p_value
    }

bp_test_result = breusch_pagan_test(results_pooled_firms, n_firms, n_years)

print("\nBreusch-Pagan LM Test for Random Effects")
print("="*70)
print(f"H0: No random effects (pooled OLS adequate)")
print(f"Ha: Random effects present\n")
print(f"LM statistic: {bp_test_result['lm_statistic']:.4f}")
print(f"P-value: {bp_test_result['p_value']:.4e}")

if bp_test_result['p_value'] < 0.05:
    print("\n✅ Reject H0: Random effects are present")
    print("   → Panel model needed (not pooled OLS)")
else:
    print("\n❌ Fail to reject H0: No evidence of random effects")
    print("   → Pooled OLS may be adequate")

---

## Exercises

### Exercise 1: Implement Manual Hausman Test

**Task:** Implement the Hausman test calculation manually to understand the mechanics.

In [ ]:
# YOUR CODE HERE
# ---------------

def manual_hausman_test(df, entity_col, time_col, y_col, x_cols):
    """
    Manually implement Hausman test.
    
    Steps:
    1. Estimate FE and RE
    2. Extract coefficients and covariances
    3. Compute test statistic
    4. Calculate p-value
    """
    # TODO: Implement manual Hausman test
    pass

manual_hausman_result = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_1():
    assert manual_hausman_result is not None, "Don't forget to implement your solution!"
    assert 'h_statistic' in manual_hausman_result, "Missing h_statistic"
    assert 'p_value' in manual_hausman_result, "Missing p_value"
    
    # Should match automated result (approximately)
    assert np.abs(manual_hausman_result['h_statistic'] - hausman_firms['test_statistic']) < 0.1, \
        "Hausman statistic doesn't match automated result"
    
    print("✅ Exercise 1 passed!")
    print("\nYour Hausman test results:")
    print(f"  H-statistic: {manual_hausman_result['h_statistic']:.4f}")
    print(f"  P-value: {manual_hausman_result['p_value']:.4f}")

# Uncomment to test
# test_exercise_1()

### Exercise 2: Model Selection Decision Tree

**Task:** Create a function that implements a complete model selection procedure.

In [ ]:
# YOUR CODE HERE
# ---------------

def model_selection_procedure(df, entity_col, time_col, y_col, x_cols, alpha=0.05):
    """
    Complete model selection procedure.
    
    Steps:
    1. F-test: Pooled vs FE
    2. BP test: Pooled vs RE
    3. If panel needed, Hausman: FE vs RE
    4. Return recommended model
    """
    # TODO: Implement model selection procedure
    pass

selection_result = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_2():
    assert selection_result is not None, "Don't forget to implement your solution!"
    assert 'recommended_model' in selection_result, "Missing recommended_model"
    assert selection_result['recommended_model'] in ['pooled', 'fe', 're'], "Invalid model recommendation"
    
    print("✅ Exercise 2 passed!")
    print("\nModel Selection Results:")
    print("="*50)
    for key, val in selection_result.items():
        print(f"{key}: {val}")

# Uncomment to test
# test_exercise_2()

### Exercise 3: Simulation Study

**Task:** Run Monte Carlo simulation to verify Hausman test properties.

In [ ]:
# YOUR CODE HERE
# ---------------

def hausman_simulation(n_sims=1000, n_entities=100, n_periods=10, correlation=0.0, alpha=0.05):
    """
    Monte Carlo simulation of Hausman test.
    
    Under H0 (correlation=0):
    - Rejection rate should be ≈ alpha (5%)
    
    Under Ha (correlation≠0):
    - Rejection rate should be high (power)
    """
    # TODO: Implement simulation
    pass

sim_results = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_3():
    assert sim_results is not None, "Don't forget to implement your solution!"
    assert 'rejection_rate' in sim_results, "Missing rejection_rate"
    
    # Under H0, rejection rate should be close to 5%
    if sim_results.get('correlation', 0) == 0:
        assert 0.03 < sim_results['rejection_rate'] < 0.07, \
            f"Rejection rate under H0 should be ~5%, got {sim_results['rejection_rate']:.1%}"
    
    print("✅ Exercise 3 passed!")
    print("\nSimulation Results:")
    print(f"  Rejection rate: {sim_results['rejection_rate']:.1%}")
    print(f"  Expected (α=0.05): 5.0%")

# Uncomment to test
# test_exercise_3()

---

## Summary

### Key Takeaways

1. **FE vs RE Trade-off:** Consistency (FE) vs efficiency (RE)

2. **Hausman Test:** Tests whether $Cov(\alpha_i, X_{it}) = 0$ holds

3. **Decision Rule:**
   - Reject Hausman → Use FE
   - Fail to reject → Use RE (more efficient)

4. **Additional Tests:**
   - F-test: Whether panel model needed
   - BP test: Evidence of random effects

5. **Practical Considerations:**
   - Economic theory should guide model choice
   - Test results are evidence, not proof
   - When in doubt, FE is safer (consistent under weaker assumptions)

### Model Selection Flowchart

```
1. F-test: Pooled vs Panel?
   ├─ Fail to reject → Pooled OLS may be adequate
   └─ Reject → Panel model needed (go to 2)

2. Hausman test: FE vs RE?
   ├─ Reject → Use Fixed Effects
   └─ Fail to reject → Use Random Effects
```

### What's Next

In the next notebook:
- Diagnostic tests for panel models
- Heteroskedasticity detection
- Serial correlation tests
- Cross-sectional dependence

---

**Next:** `02_diagnostics.ipynb` - Panel Model Diagnostics

In [ ]:
key_takeaways(["Next:"])